# Module 2 -- Step 3: Virtual Keys and Teams

This notebook sets up virtual keys and teams in the LiteLLM Proxy interactively.
It performs the same operations as `scripts/setup_keys.py` but step by step,
so you can inspect the results at each stage.

**Prerequisites:**
- Step 2 completed (stack deployed, `.state.json` exists)
- Gateway is healthy and reachable

## Install Dependencies

## Load State from Previous Step

In [ ]:
import json, time, boto3, requests

with open(".state.json") as f:
    state = json.load(f)

LLM_GATEWAY_URL = state["LLM_GATEWAY_URL"]
ADMIN_KEY = state["ADMIN_KEY"]
AWS_REGION = state["AWS_REGION"]
HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {ADMIN_KEY}"}
print(f"Gateway URL: {LLM_GATEWAY_URL}")

## Register Bedrock Models

LiteLLM needs model entries in its database before virtual keys can reference them.
Each entry maps a short alias (e.g. `claude-sonnet`) to its full Bedrock model ID
so callers only need the alias.

We register a curated set of models covering Anthropic Claude, Amazon Nova,
Meta Llama, and DeepSeek families. The catalog is **region-independent**: each
entry stores a bare model suffix (no geo/`global.` prefix, no `bedrock/`
segment) plus two flags. We build the concrete model id at registration time
from the deploy region, mirroring `scripts/setup_keys.py`:

- Claude 4.x and Nova-2 use the region-agnostic `global.` inference profile.
- Older Nova v1, Llama, and DeepSeek use a geo-scoped profile (`us.`/`eu.`/`apac.`)
  derived from the region.
- Bare foundation models take no prefix.

In [ ]:
import os

# --- bedrock_region mirror (kept in sync with scripts/bedrock_region.py) ---
# Map of region prefix -> Bedrock geo inference-profile prefix.
# Canada (ca-) and South America (sa-) route via the US geo today.
# NOTE: Asia-Pacific (ap-), Middle East (me-) and Africa (af-) all use "apac." (not "ap.").
_GEO_MAP = {"us": "us.", "ca": "us.", "sa": "us.", "eu": "eu.", "ap": "apac.", "me": "apac.", "af": "apac."}

def geo_prefix(region):
    return _GEO_MAP.get((region or "").split("-")[0], "us.")

def model_id(suffix, region, needs_profile=True, prefer_global=False):
    """Build a Bedrock model id from a bare suffix (no geo/global prefix, no 'bedrock/')."""
    if prefer_global:
        return "bedrock/global." + suffix
    if needs_profile:
        return "bedrock/" + geo_prefix(region) + suffix
    return "bedrock/" + suffix

# Resolve the region: prefer the value loaded from .state.json, then the
# standard env vars / boto3 session. Refuse ONLY when empty.
REGION = AWS_REGION or os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or boto3.Session().region_name
if not REGION:
    raise RuntimeError(
        "Could not resolve an AWS region. Set AWS_REGION / AWS_DEFAULT_REGION "
        "or configure a default region (e.g. `aws configure set region <region>`)."
    )
AWS_REGION = REGION

# Region-independent catalog: (alias, bare_suffix, needs_profile, prefer_global).
WORKSHOP_MODELS = [
    # Anthropic Claude 4.x — global. inference profile (region-agnostic)
    ("claude-sonnet", "anthropic.claude-sonnet-4-6", True, True),
    ("claude-haiku", "anthropic.claude-haiku-4-5-20251001-v1:0", True, True),
    ("claude-sonnet-4.6", "anthropic.claude-sonnet-4-6", True, True),
    ("claude-sonnet-4.5", "anthropic.claude-sonnet-4-5-20250929-v1:0", True, True),
    ("claude-haiku-4.5", "anthropic.claude-haiku-4-5-20251001-v1:0", True, True),
    # Amazon Nova — Nova-2 uses global.; older Nova v1 uses geo-scoped profiles
    ("nova-premier", "amazon.nova-premier-v1:0", True, False),
    ("nova-pro", "amazon.nova-pro-v1:0", True, False),
    ("nova-lite", "amazon.nova-lite-v1:0", True, False),
    ("nova-2-lite", "amazon.nova-2-lite-v1:0", True, True),
    # DeepSeek — geo-scoped inference profile
    ("deepseek-r1", "deepseek.r1-v1:0", True, False),
]

# Clean up existing models first — narrow to HTTP errors so auth/network failures surface
try:
    resp = requests.get(f"{LLM_GATEWAY_URL}/model/info", headers=HEADERS, timeout=10)
    if resp.ok:
        for model in resp.json().get("data", []):
            model_info_id = model.get("model_info", {}).get("id", "")
            if model_info_id:
                try:
                    requests.post(f"{LLM_GATEWAY_URL}/model/delete", json={"id": model_info_id}, headers=HEADERS, timeout=10)
                except requests.RequestException as e:
                    print(f"  Warning: failed to delete {model_info_id}: {e}")
        print(f"Cleaned up existing model entries.")
except requests.RequestException as e:
    print(f"Warning: could not list existing models ({e}); continuing with registration.")

# Register models — build the region-correct id at registration time
registered = 0
first_error = None
for model_name, suffix, needs_profile, prefer_global in WORKSHOP_MODELS:
    litellm_model = model_id(suffix, AWS_REGION, needs_profile=needs_profile, prefer_global=prefer_global)
    try:
        resp = requests.post(f"{LLM_GATEWAY_URL}/model/new", json={
            "model_name": model_name,
            "litellm_params": {"model": litellm_model, "aws_region_name": AWS_REGION},
        }, headers=HEADERS, timeout=10)
        if resp.ok or (resp.status_code == 400 and "already exists" in resp.text.lower()):
            registered += 1
        else:
            first_error = first_error or f"{model_name}: {resp.status_code} {resp.text[:200]}"
            print(f"  Warning: {model_name}: {resp.status_code} {resp.text[:200]}")
    except requests.RequestException as e:
        first_error = first_error or f"{model_name}: {e}"
        print(f"  Warning: {model_name}: {e}")

print(f"\nRegistered {registered}/{len(WORKSHOP_MODELS)} models")
if registered == 0:
    raise RuntimeError(
        f"No models registered — check Bedrock model access in the AWS console "
        f"(Bedrock > Model access). First error: {first_error}"
    )

## Create Teams

Teams provide budget governance boundaries. Each team has its own spending limit
and a list of models its members are allowed to use.

We create two teams:
- **platform-team** -- higher budget ($10), for platform engineers
- **workload-team** -- lower budget ($5), for application developers

In [ ]:
# Idempotent: list existing teams and reuse when alias matches
teams = {}
existing_teams = []
try:
    list_resp = requests.get(f"{LLM_GATEWAY_URL}/team/list", headers=HEADERS, timeout=10)
    if list_resp.ok:
        body = list_resp.json()
        existing_teams = body if isinstance(body, list) else body.get("teams", body.get("data", []))
except requests.RequestException:
    existing_teams = []

for alias, budget in [("platform-team", 10.0), ("workload-team", 5.0)]:
    match = next((t for t in existing_teams if t.get("team_alias") == alias), None)
    if match and match.get("team_id"):
        teams[alias] = match["team_id"]
        print(f"Reusing existing team '{alias}' (id={teams[alias][:12]}..., budget=${budget})")
        continue
    resp = requests.post(f"{LLM_GATEWAY_URL}/team/new", json={
        "team_alias": alias,
        "max_budget": budget,
        "models": ["claude-sonnet", "claude-haiku", "nova-2-lite"],
    }, headers=HEADERS, timeout=10)
    if not resp.ok:
        raise RuntimeError(f"Team create for {alias} failed: {resp.status_code} {resp.text[:300]}")
    team_data = resp.json()
    teams[alias] = team_data.get("team_id", "")
    print(f"Created team '{alias}' (id={teams[alias][:12]}..., budget=${budget})")

## Create Virtual Keys

Virtual keys are issued to individual users or services. Each key is scoped to a
team and carries its own budget cap. When either the key budget or the team budget
is exhausted, requests are rejected -- this gives fine-grained cost control.

We create two keys:
- **workshop-admin-key** -- belongs to platform-team ($10 budget)
- **workshop-dev-key** -- belongs to workload-team ($5 budget)

In [ ]:
# Idempotent: reuse stored keys from .state.json if present; else list+match; else create
keys = dict(state.get("VIRTUAL_KEYS", {}))
for key_name, team_alias, budget in [("workshop-admin-key", "platform-team", 10.0), ("workshop-dev-key", "workload-team", 5.0)]:
    if keys.get(key_name):
        # Validate the stored key still exists
        info = requests.get(f"{LLM_GATEWAY_URL}/key/info", headers=HEADERS, params={"key": keys[key_name]}, timeout=10)
        if info.ok:
            print(f"Reusing stored key '{key_name}' = {keys[key_name][:16]}...")
            continue
        print(f"Stored key '{key_name}' is no longer valid; recreating.")
    resp = requests.post(f"{LLM_GATEWAY_URL}/key/generate", json={
        "key_name": key_name,
        "team_id": teams[team_alias],
        "max_budget": budget,
        "models": ["claude-sonnet", "claude-haiku", "nova-2-lite"],
    }, headers=HEADERS, timeout=10)
    if not resp.ok:
        raise RuntimeError(f"Key generate for {key_name} failed: {resp.status_code} {resp.text[:300]}")
    key_data = resp.json()
    keys[key_name] = key_data.get("key", "")
    print(f"Created key '{key_name}' = {keys[key_name][:16]}... (budget=${budget})")

API_KEY = keys["workshop-dev-key"]

## Test Chat Completion

Verify the virtual key works end-to-end by sending a chat completion request
through the gateway to Bedrock.

In [ ]:
test_headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
resp = requests.post(f"{LLM_GATEWAY_URL}/chat/completions", json={
    "model": "claude-sonnet",
    "messages": [{"role": "user", "content": "Say hello in exactly 5 words."}],
    "max_tokens": 50, "temperature": 0.5,
}, headers=test_headers, timeout=30)
if not resp.ok:
    raise RuntimeError(f"Chat completion failed: {resp.status_code} {resp.text[:400]}")
content = resp.json()["choices"][0]["message"]["content"]
print(f"Model response: {content}")
print("\nBedrock is working through the LiteLLM Proxy!")

## Query Key Info

Inspect the virtual key metadata, including spend so far and remaining budget.

In [ ]:
resp = requests.get(f"{LLM_GATEWAY_URL}/key/info", headers=HEADERS, params={"key": API_KEY}, timeout=10)
info = resp.json()
# Redact token/key values before printing
def _redact(obj):
    if isinstance(obj, dict):
        return {k: ("***redacted***" if k in ("token", "key") and isinstance(v, str) else _redact(v)) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact(x) for x in obj]
    return obj
print(json.dumps(_redact(info), indent=2, default=str))

## Save State

Persist the API key into `.state.json` so subsequent steps can use it.

In [ ]:
state["API_KEY"] = API_KEY
state["ADMIN_KEY"] = ADMIN_KEY
state["VIRTUAL_KEYS"] = keys  # persist for idempotent rerun
state["TEAMS"] = teams
with open(".state.json", "w") as f:
    json.dump(state, f, indent=2)
print("State saved (API_KEY + VIRTUAL_KEYS + TEAMS added)")

## Next Step

Proceed to **Step 4** to explore the LLM Gateway walkthrough -- multi-model routing,
caching, spend tracking, and Strands Agent integration.